# Package 설치

In [ ]:
!uv pip install ipykernel ipywidgets
# !uv pip install torch==2.8
# !uv pip install transformers tokenizers datasets accelerate sentencepiece pillow  timm

Audited 7 packages in 14ms


# HuggingFace transformers의 Pipeline을 이용한 모델 활용

- Pipeline은 Transformers 라이브러리의 가장 기본적인 객체로, **전처리 - 추론 -> 후처리** 로 이어지는 일련의 과정을 자동화하여 손쉽게 모델을 사용할 수 있게 해준다.
- Task에 따라 다양한 Pipeline 클래스를 제공하며 `pipeline` 함수를 이용해 쉽게 생성할 수 있다.
- **task만 지정**해서 그 task에 대해 기본적으로 제공 모델과 토크나이저를 사용하거나 또는 **직접 사용할 모델과 토크나이저를 지정**해 생성할 수 있다.
  - **토크나이저의 경우 같은 사용할 모델의 ID로 Load하여 그 모델이 학습할 때 사용한 것을 로드한다.**
- https://huggingface.co/docs/transformers/pipeline_tutorial

![huggingface_pipeline.png](figures/huggingface_pipeline.png)

## 지원하는 주요 태스크
- https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.pipeline.task
### 자연어 처리 태스크
- **text-classification**: 텍스트 분류
- **text-generation**: 텍스트 생성
- **translation**: 번역
- **summarization**: 요약
- **question-answering**: 질의응답
- **fill-mask**: 마스크 토큰 채우기
- **token-classification**: 개체명 인식, Pos tagging 같이 개별 토큰에 대한 분류
- **feature-extraction**: 특징 추출(context vector)

### 영상 처리 태스크
- **image-classification**: 이미지 분류
- **object-detection**
  -  객체 검출 (Object Detection)
  -  이미지 안에서 객체들의 위치와 class를 찾아내는 작업
- **image-segmentation**
  -  이미지 세분화 (Image Segmentation)
  -  이미지를 픽셀 단위로 분할하여 각 픽셀이 어떤 객체에 속하는지 분류하는 작업

## 모델 검색
![huggingface_model_search.png](figures/huggingface_model_search.png)



## pipeline 함수
- 주요파라미터
  - **task:** 수행하려는 작업의 유형을 문자열로 지정한다.
  - **model:**
    - 사용할 사전 학습된 모델의 이름 또는 경로를 지정한다.
    - 모델이름(ID)은 `[모델소유자이름]/[모델이름]` 형식이다. Hugging Face에서 제공하는 모델의 경우는 `모델소유자이름`이 생략되어 있다. (ex: "google/gemma-2-2b", "gpt2")
    - 모델을 명시적으로 지정하지 않으면, **task에 맞는 기본 모델이 로드**된다.
  - **tokenizer:** 자연어 task에서 사용할 토크나이저를 지정한다. 생략하면 모델과 같이 제공되는(model과 이름이 같은 토크나이저) 토크나이저를 사용한다.
  - **framework:** 사용할 딥러닝 프레임워크를 지정한다. 'pt'는 PyTorch(Default), 'tf'는 TensorFlow를 지정한다.
  - **device:** Pipeline 모델을 실행할 디바이스를 지정한다. 문자열로 `"cpu", "cuda:1", "mps"`, 또는 GPU 번호를 정수로 지정한다.
  - **revision:** 모델의 특정 버전을 지정할 때 사용한다.
  - **trust_remote_code:** hub 모델을 직접 다운 받는 것이 아니라 모델을 다운 받는 **코드**를 다운 받아 local에서 실행하는 경우 코드를 실행할 수있게 할 지 여부. (bool)
  - **use_fast:**
    - 빠른 토크나이저를 사용할지 여부를 지정합니다. 기본값은 True입니다.
    - 빠른 토크나이저는 `Rust` 언어로 구현되어 속도가 빠르다. 단 모든 모델에 대해 지원하지 않는다. 지원하지 않을 경우 `use_fast=True`로 설정해도 일반 토크나이저가 사용된다.

In [1]:
import transformers

transformers.__version__

'4.57.3'

## Task 별 pipeline 실습

### 텍스트 분류

In [ ]:
from transformers import pipeline

# task만 지정 : 그 task를 실행할 수 있는 기본 모델과 토크나이저를 이용해서 pipline 생성.
pipe = pipeline(task = "text-classification")
# 긍정, 부정 분류만 존재하는 것 아님 → 화남, 행복 등의 분류도 있음

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


In [ ]:
# 모델을 저장할 local repository
# c:\사용자\playdata\.cache/huggingface/hub

In [ ]:
result = pipe("I am very happy")

In [ ]:
result

[{'label': 'POSITIVE', 'score': 0.9998795986175537}]

In [ ]:
data = [
    "The project was completed successfully.",
    "She always brings positive energy to the team.",
    "I am confident that we will achieve our goals.",
    "The results were not as expected.",
    "He struggled to meet the deadline.",
    "The client was dissatisfied with the final product."
]


In [ ]:
result = pipe(data)

In [ ]:
result

[{'label': 'POSITIVE', 'score': 0.9998227953910828},
 {'label': 'POSITIVE', 'score': 0.9998812675476074},
 {'label': 'POSITIVE', 'score': 0.9998470544815063},
 {'label': 'NEGATIVE', 'score': 0.9978100657463074},
 {'label': 'NEGATIVE', 'score': 0.99960857629776},
 {'label': 'NEGATIVE', 'score': 0.9996129870414734}]

In [ ]:
# pipeline 생성 시 모델을 지정.
# 1. model_id 지정 → 지정한 ID의 모델과 토크나이저로 pipeline을 구성
# 2. model과 tokenizer(전처리)를 직접 생성해서 pipeline에 넣어 생성.

model = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
pipe = pipeline(
    task = "text-classification",
    model=model
    # tokenizer = model # tokenizer와 model의 id가 같은 경우 생략.
)

Device set to use cpu


In [ ]:
pipe(data)

[{'label': 'POSITIVE', 'score': 0.9998227953910828},
 {'label': 'POSITIVE', 'score': 0.9998812675476074},
 {'label': 'POSITIVE', 'score': 0.9998470544815063},
 {'label': 'NEGATIVE', 'score': 0.9978100657463074},
 {'label': 'NEGATIVE', 'score': 0.99960857629776},
 {'label': 'NEGATIVE', 'score': 0.9996129870414734}]

In [ ]:
kor_texts = [
    "이 영화 정말 재미있어요!",
    "서비스가 별로였어요.",
    "제품 품질이 우수합니다.",
    "따듯하고 부드럽고 제품은 너무 좋습니다. 그런데 배송이 너무 늦네요."
]

In [ ]:
model = "Copycats/koelectra-base-v3-generalized-sentiment-analysis"
pipe = pipeline(task="text-classification", model=model)

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

c:\Users\Playdata\Documents\SKN21\JYS\09_Huggingface_transformers\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--Copycats--koelectra-base-v3-generalized-sentiment-analysis. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo

model.safetensors:   0%|          | 0.00/452M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


In [ ]:
pipe(kor_texts)

[{'label': '1', 'score': 0.9897311329841614},
 {'label': '0', 'score': 0.9969298243522644},
 {'label': '1', 'score': 0.9640172123908997},
 {'label': '0', 'score': 0.5669127702713013}]

### 제로샷 분류
- 제로샷(Zero-shot)은 각 개별 작업에 대한 특정 교육 없이 작업을 수행할 수 있는 task다.
- 입력 텍스트와 함께 클래스 레이블을 제공하면 분류 작업을 한다.
- 모델은  `task`에서 `Zero-Shot` 으로 시작하는 task를 선택하여 검색한다.

In [ ]:
model = "facebook/bart-large-mnli"
pipe = pipeline(task="zero-shot-classification", model=model)

config.json: 0.00B [00:00, ?B/s]

c:\Users\Playdata\Documents\SKN21\JYS\09_Huggingface_transformers\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [ ]:
text = ["Python is a programming language.",
        "I love soccer",
        "The stock price rose slightly today."]

labels1 = ["IT", "Sports"] # IT와 Sports로 분류해달라.
labels2 = ["business", "programming", "sports", "movie", "education"] #

# 분류할 대상과 분류index를 나눠주는 것을 제공. > 어떻게 분류할지가 미리 학습된 상태가 아닌 것임.


In [ ]:
result = pipe(text, candidate_labels = labels1) # 후보라벨은 labels1.
result

[{'sequence': 'Python is a programming language.',
  'labels': ['IT', 'Sports'],
  'scores': [0.5758535265922546, 0.4241464138031006]},
 {'sequence': 'I love soccer',
  'labels': ['Sports', 'IT'],
  'scores': [0.9935312867164612, 0.006468690931797028]},
 {'sequence': 'The stock price rose slightly today.',
  'labels': ['IT', 'Sports'],
  'scores': [0.6849520802497864, 0.3150479197502136]}]

In [ ]:
result = pipe(text, candidate_labels = labels2)
result

[{'sequence': 'Python is a programming language.',
  'labels': ['programming', 'business', 'movie', 'sports', 'education'],
  'scores': [0.9856367111206055,
   0.005072721280157566,
   0.0034023483749479055,
   0.002961924998089671,
   0.0029262355528771877]},
 {'sequence': 'I love soccer',
  'labels': ['sports', 'programming', 'business', 'movie', 'education'],
  'scores': [0.9952405691146851,
   0.0012840895215049386,
   0.0012676474871113896,
   0.0012649551499634981,
   0.0009427034528926015]},
 {'sequence': 'The stock price rose slightly today.',
  'labels': ['business', 'movie', 'programming', 'sports', 'education'],
  'scores': [0.7462778091430664,
   0.06974831968545914,
   0.06889291107654572,
   0.0645080953836441,
   0.05057287961244583]}]

### 텍스트 생성

In [ ]:
pipe = pipeline(task = "text-generation")

# 문장이 오면 이어지는 문장을 만들어줌

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d (https://huggingface.co/openai-community/gpt2).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

c:\Users\Playdata\Documents\SKN21\JYS\09_Huggingface_transformers\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--openai-community--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not in

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [ ]:
result=pipe("Python is a")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [ ]:
result

[{'generated_text': 'Python is a program that is written in Python. It is the first of the type-safe language constructs designed for programming in C. The object of this type is a collection of integers that represent various bits in the memory of the computer. Each integer is represented by a specific function (the basic form of the object). The object is stored in a collection called a list. The function call (in the form of a single argument) is the name of the function that contains the integer. The function call value can be any integer (a, b, c) or an integer (a, b, c). The functions are called with a single argument. The argument is an integer. The object is initialized by calling the default constructor with a new object of the form the following:\n\nclass List { def __init__(self, value_type): self.value_type = value_type self.value_type += value_type return List() }\n\nThe default constructor is called after the list has been initialized. The default constructor is a functio

In [ ]:
print(result[0]['generated_text'])

Python is a program that is written in Python. It is the first of the type-safe language constructs designed for programming in C. The object of this type is a collection of integers that represent various bits in the memory of the computer. Each integer is represented by a specific function (the basic form of the object). The object is stored in a collection called a list. The function call (in the form of a single argument) is the name of the function that contains the integer. The function call value can be any integer (a, b, c) or an integer (a, b, c). The functions are called with a single argument. The argument is an integer. The object is initialized by calling the default constructor with a new object of the form the following:

class List { def __init__(self, value_type): self.value_type = value_type self.value_type += value_type return List() }

The default constructor is called after the list has been initialized. The default constructor is a function with a single argument.

In [ ]:
pipe("나는 어제")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': '나는 어제만는\n\n방방서 보렇어가기\n\n연는 주면요\n\n쁡아지 우에 지 우에는\n\n가 원을 오지 우에는\n\n지 우에는 일 에에는\n\n연는 주면요\n\n상한 사랑우\n\n방서 보렇어가기\n\n어제만는 어제만는에는\n\n\n자는 게요요\n\n\n주면요\n\n\n아나렌'}]

In [ ]:
model = 'Qwen/Qwen3-0.6B'
pipe = pipeline("text-generation", model=model)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

c:\Users\Playdata\Documents\SKN21\JYS\09_Huggingface_transformers\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--Qwen--Qwen3-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installe

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
result = pipe("나는 어제")

In [ ]:
print(result[0]['generated_text'])

나는 어제 끝났다. 어제는 제 뷰티 앱에서 끝났다. 그리고 그 시간은 매우 흥미로운 일었어. 나는 끝났다. 오늘은 첫 번째 일이다. 그리고 나중에 일가지에서 끝날 수 있다. 오늘은 힙스케이프 일이다.

So, the story is about the person who has finished their first day at a company. The person's first day was at their beauty app. The person found the time very interesting. The person then ended the day. The person is now in the morning. The story is about the person who has finished their first day at a company. The person is now in the morning.

The story is about the person who has finished their first day at a company. The person is now in the morning.

The story is about the person who has finished their first day at a company. The person is now in the morning.

The story is about the person who has finished their first day at a company. The person is now in the morning.

The story is about the person who has finished their first day at a company. The person is now in the morning.

The story is


### 마스크 채우기

In [ ]:
text = "I'm going to <mask> because <mask> am hurt."


In [ ]:
model = 'FacebookAI/xlm-roberta-base'

pipe = pipeline("fill-mask", model=model)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

c:\Users\Playdata\Documents\SKN21\JYS\09_Huggingface_transformers\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--FacebookAI--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is 

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of the model checkpoint at FacebookAI/xlm-roberta-base were not used when initializing XLMRobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [ ]:
result = pipe(text)

In [ ]:
len(result)

2

In [ ]:
len(result[0]) # 첫번째 <mask>부분에 들어갈 가능성이 높은 단어 5개를 보여줌

for r in result[0]:
    print(r)

{'score': 0.41619813442230225, 'token': 91190, 'token_str': 'cry', 'sequence': "<s> I'm going to cry because<mask> am hurt.</s>"}
{'score': 0.1767032891511917, 'token': 68, 'token_str': 'die', 'sequence': "<s> I'm going to die because<mask> am hurt.</s>"}
{'score': 0.07769551873207092, 'token': 31358, 'token_str': 'leave', 'sequence': "<s> I'm going to leave because<mask> am hurt.</s>"}
{'score': 0.04653538390994072, 'token': 738, 'token_str': 'go', 'sequence': "<s> I'm going to go because<mask> am hurt.</s>"}
{'score': 0.027225980535149574, 'token': 7279, 'token_str': 'stop', 'sequence': "<s> I'm going to stop because<mask> am hurt.</s>"}


In [ ]:
len(result[1]) # 두번째 <mask>부분에 들어갈 가능성이 높은 단어 5개를 보여줌

for r in result[1]:
    print(r)

{'score': 0.9929074048995972, 'token': 87, 'token_str': 'I', 'sequence': "<s> I'm going to<mask> because I am hurt.</s>"}
{'score': 0.006063544657081366, 'token': 17, 'token_str': 'i', 'sequence': "<s> I'm going to<mask> because i am hurt.</s>"}
{'score': 0.00029492215253412724, 'token': 398, 'token_str': 'you', 'sequence': "<s> I'm going to<mask> because you am hurt.</s>"}
{'score': 0.00010049621050711721, 'token': 442, 'token_str': 'it', 'sequence': "<s> I'm going to<mask> because it am hurt.</s>"}
{'score': 8.748131222091615e-05, 'token': 2412, 'token_str': 'she', 'sequence': "<s> I'm going to<mask> because she am hurt.</s>"}


In [ ]:
kor_text = "오늘 밤은 전국이 흐린 가운데 대부분 지역에 <mask>가 내리겠고, 기온이 내려가면서 점차 <mask>이 오는 곳이 많겠습니다"

In [ ]:
result = pipe(kor_text,top_k=2)

In [ ]:
result

[[{'score': 0.8498733639717102,
   'token': 7091,
   'token_str': '비',
   'sequence': '<s> 오늘 밤은 전국이 흐린 가운데 대부분 지역에 비 가 내리겠고, 기온이 내려가면서 점차<mask> 이 오는 곳이 많겠습니다</s>'},
  {'score': 0.10534238070249557,
   'token': 34565,
   'token_str': '눈',
   'sequence': '<s> 오늘 밤은 전국이 흐린 가운데 대부분 지역에 눈 가 내리겠고, 기온이 내려가면서 점차<mask> 이 오는 곳이 많겠습니다</s>'}],
 [{'score': 0.6368756890296936,
   'token': 34565,
   'token_str': '눈',
   'sequence': '<s> 오늘 밤은 전국이 흐린 가운데 대부분 지역에<mask> 가 내리겠고, 기온이 내려가면서 점차 눈 이 오는 곳이 많겠습니다</s>'},
  {'score': 0.14748039841651917,
   'token': 208400,
   'token_str': '구름',
   'sequence': '<s> 오늘 밤은 전국이 흐린 가운데 대부분 지역에<mask> 가 내리겠고, 기온이 내려가면서 점차 구름 이 오는 곳이 많겠습니다</s>'}]]

### Token별 분류
- task: token-classification
  - 개체명인식(ner), 품사부착(pos tagging)을 수행하는 task
  - 개체명 인식은 문장에서 특정한 개체명(예: 사람 이름, 지명, 조직명 등)을 식별하는 task이다.

In [3]:
from transformers import pipeline

In [2]:
text = "My name is Sylvain and I work at Hugging Face in Brooklyn."

In [3]:
pipe = pipeline(task = "token-classification",model = 'dbmdz/bert-large-cased-finetuned-conll03-english')
result = pipe(text)

config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [4]:
result

[{'entity': 'I-PER',
  'score': np.float32(0.99938285),
  'index': 4,
  'word': 'S',
  'start': 11,
  'end': 12},
 {'entity': 'I-PER',
  'score': np.float32(0.99815494),
  'index': 5,
  'word': '##yl',
  'start': 12,
  'end': 14},
 {'entity': 'I-PER',
  'score': np.float32(0.9959072),
  'index': 6,
  'word': '##va',
  'start': 14,
  'end': 16},
 {'entity': 'I-PER',
  'score': np.float32(0.99923277),
  'index': 7,
  'word': '##in',
  'start': 16,
  'end': 18},
 {'entity': 'I-ORG',
  'score': np.float32(0.9738931),
  'index': 12,
  'word': 'Hu',
  'start': 33,
  'end': 35},
 {'entity': 'I-ORG',
  'score': np.float32(0.976115),
  'index': 13,
  'word': '##gging',
  'start': 35,
  'end': 40},
 {'entity': 'I-ORG',
  'score': np.float32(0.9887976),
  'index': 14,
  'word': 'Face',
  'start': 41,
  'end': 45},
 {'entity': 'I-LOC',
  'score': np.float32(0.9932106),
  'index': 16,
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

### 질의 응답
- 문서와 질문을 주면 문서에서 답을 찾아 응답한다.

In [4]:
model = 'timpal0l/mdeberta-v3-base-squad2'

pipe = pipeline(task = "question-answering", model=model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/453 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
# question : 질문, context : 답을 찾을 문서

In [11]:
# question="Where do I work?"
# question="Where is Hugging Face?"
question = "What is my mane?"
context="My name is Sylvain and I work at Hugging Face in Brooklyn"

In [12]:
result = pipe(question=question, context=context)

In [13]:
result

{'score': 0.2789620263502002, 'start': 10, 'end': 18, 'answer': ' Sylvain'}

In [14]:
context = """우리나라 2대 수출 품목인 자동차가 도널드 트럼프 미국 행정부의 관세 여파로 지난달 큰 폭의 수출 감소율을 보이면서 우려가 커지고 있다. 현대차, 기아의 미국 수출 비중이 최대 85%에 이르는 상황에서 자동차 관세 장기화 시 피해는 걷잡을 수 없이 불어날 것이라는 암울한 전망이 나온다.
1일 산업통상자원부가 발표한 5월 수출입 동향에 따르면 지난달 자동차 수출은 작년 동기 대비 4.4% 감소한 62억달러로 집계됐다. 최대 자동차 시장인 미국으로의 수출은 18억4000만달러로 무려 32.0% 급감했다.
4월 미국의 수입산 자동차 25% 관세 부과에 이어 5월부터 일부 자동차 부품에도 25%의 관세가 적용된 결과다. 관세 장기화 시 피해는 더 커질 것이라는 우려가 현실화한 셈이다.
국내 완성차 1·2위 업체인 현대차·기아는 현지 생산 비중을 확대하는 동시에 가격 인상을 검토하고 있다. 관세 여파를 흡수하기 위해서다. 가격 인상이 현실화할 경우 미국 현지 판매는 줄어들 수밖에 없어 수출에는 더 악영향을 미칠 것으로 보인다.
"""

q1 = "현대차 기아의 미국 수출비중은?"
q2 = "자동차 수출이 얼마나 급감했나?"
q3 = "대미 수출 감소에 국내 자동차 업체들의 대응방법은?"

In [15]:
result = pipe(question=[q1,q2,q3], context=context)

[{'score': 0.4708772700978443, 'start': 95, 'end': 103, 'answer': ' 최대 85%에'},
 {'score': 0.9133427568594925, 'start': 270, 'end': 276, 'answer': ' 32.0%'},
 {'score': 0.2734988257288933,
  'start': 426,
  'end': 442,
  'answer': ' 가격 인상을 검토하고 있다.'}]

In [16]:
len(result)

3

In [17]:
for r in result :
  print(r)

{'score': 0.4708772700978443, 'start': 95, 'end': 103, 'answer': ' 최대 85%에'}
{'score': 0.9133427568594925, 'start': 270, 'end': 276, 'answer': ' 32.0%'}
{'score': 0.2734988257288933, 'start': 426, 'end': 442, 'answer': ' 가격 인상을 검토하고 있다.'}


In [20]:
context[95:103]

' 최대 85%에'

### 문서 요약

In [21]:
model = 'eenzeenee/t5-base-korean-summarization'
pipe = pipeline(task="summarization", model=model)

config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [22]:
result = pipe(context)

Token indices sequence length is longer than the specified maximum sequence length for this model (368 > 128). Running this sequence through the model will result in indexing errors


In [24]:
result[0]['summary_text']
# 요약된 문장이 나옴

'자동차가 트럼프 미국 행정부의 관세 여파로 큰 폭의 수출 감소율을 보이면서 자동차 관세 장기화 시 피해는 걷잡을 수 없이 불어날 것이라는 암울한 전망이 나온다.'

### 번역

In [25]:
text = "Ce cours est produit par Hugging Face."

In [27]:
model = 'Helsinki-NLP/opus-mt-fr-en' # fr : 불어를 en: 영어로 번역.
pipe = pipeline(task="translation",model=model)

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu


In [ ]:
!uv pip install sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 10.5 MB/s eta 0:00:00


In [29]:
result = pipe(text)
result

[{'translation_text': 'This course is produced by Hugging Face.'}]

In [31]:
print(text)
print(result[0]['translation_text'])

Ce cours est produit par Hugging Face.
This course is produced by Hugging Face.


In [32]:
text_list = ["이 문장을 영어로 번역합니다.", "날씨가 점점 더워집니다.", "오늘 비가 올 것 같습니다."]

In [33]:
pipe(text_list)
# 프랑스어 > 영어 번역하는 모델이기 때문에 한국어를 넣으면 답변이 상관없는 답변이 나옴.

[{'translation_text': "- I don't know. - I don't know. - I don't know. - I don't know."},
 {'translation_text': '- Yeah. - Yeah. - Yeah, yeah.'},
 {'translation_text': "- I don't know. - I don't know. - I don't know. - I don't know."}]

In [34]:
model = 'Helsinki-NLP/opus-mt-ko-en'
pipe = pipeline(task="translation",model=model)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/842k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [35]:
result = pipe(text_list)
result

[{'translation_text': 'I translate this sentence into English.'},
 {'translation_text': 'The weather gets warmer and warmer.'},
 {'translation_text': "It's going to rain today."}]

### 이미지를 설명하는 텍스트 생성

In [38]:
url1 = "https://huggingface.co/datasets/Narsil/image_dummy/resolve/main/parrots.png"
url2 = "https://th.bing.com/th?id=ORMS.c526884bbea37c0bb9501f4f83b601e4&pid=Wdp&w=268&h=140&qlt=90&c=1&rs=1&dpr=1&p=0"
url3 = "http://images.cocodataset.org/val2017/000000039769.jpg"

In [36]:
model = 'Salesforce/blip-image-captioning-base'
pipe = pipeline(task="image-to-text", model=model)

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cpu


In [39]:
result = pipe(url1)
result

[{'generated_text': 'two birds are standing next to each other birds'}]

In [40]:
result = pipe([url1,url2, url3])

result

[[{'generated_text': 'two birds are standing next to each other birds'}],
 [{'generated_text': 'a baseball player is throwing a pitch'}],
 [{'generated_text': 'two cats sleeping on a couch'}]]

In [43]:
result=pipe("/content/3c52bb2ffbc57ffe.png")
result

[{'generated_text': 'a small dog is sitting on a pillow'}]

### 이미지 분류

In [44]:
url = "https://pds.joongang.co.kr/news/component/htmlphoto_mmdata/202306/25/488f9638-800c-4bac-ad65-82877fbff79b.jpg"

In [45]:
model = 'google/vit-base-patch16-224'
pipe = pipeline(task="image-classification", model=model)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Device set to use cpu


In [47]:
result = pipe(url)
result

[{'label': 'Egyptian cat', 'score': 0.8531320691108704},
 {'label': 'tabby, tabby cat', 'score': 0.047503720968961716},
 {'label': 'tiger cat', 'score': 0.034866124391555786},
 {'label': 'Persian cat', 'score': 0.007555840071290731},
 {'label': 'Siamese cat, Siamese', 'score': 0.003788585541769862}]

In [49]:
result = pipe([url, 'images.jpg','3c52bb2ffbc57ffe.png'], top_k=2)

for r in result :
  print(r)

[{'label': 'Egyptian cat', 'score': 0.8531320691108704}, {'label': 'tabby, tabby cat', 'score': 0.047503720968961716}]
[{'label': 'nipple', 'score': 0.026889940723776817}, {'label': 'piggy bank, penny bank', 'score': 0.024660784751176834}]
[{'label': 'Chihuahua', 'score': 0.7935053110122681}, {'label': 'toy terrier', 'score': 0.160121887922287}]


In [50]:
pipe(['data/image1.jpg','data/image3.jpg'],top_k=2)

[[{'label': 'Old English sheepdog, bobtail', 'score': 0.9108919501304626},
  {'label': 'komondor', 'score': 0.07120531797409058}],
 [{'label': 'desktop computer', 'score': 0.84475177526474},
  {'label': 'screen, CRT screen', 'score': 0.07074463367462158}]]

### Object Detection

In [58]:
image_path1 = r"data/image1.jpg"
image_path2 = r"data/image2.jpg"
image_path3 = r"data/image3.jpg"

In [51]:
pipe = pipeline(task="object-detection")

No model was supplied, defaulted to facebook/detr-resnet-50 and revision 1d5f47b (https://huggingface.co/facebook/detr-resnet-50).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/167M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2446: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2446: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2446: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pas

preprocessor_config.json:   0%|          | 0.00/290 [00:00<?, ?B/s]

Device set to use cpu


In [59]:
result = pipe([image_path1, image_path2, image_path3])

In [57]:
result

[{'score': 0.9956639409065247,
  'label': 'cell phone',
  'box': {'xmin': 96, 'ymin': 165, 'xmax': 136, 'ymax': 236}},
 {'score': 0.9919517040252686,
  'label': 'tv',
  'box': {'xmin': 147, 'ymin': 28, 'xmax': 429, 'ymax': 240}},
 {'score': 0.9975364208221436,
  'label': 'keyboard',
  'box': {'xmin': 108, 'ymin': 251, 'xmax': 358, 'ymax': 304}}]

In [ ]:
# 박스형태로 물체의 위치를 나타내줌.
# min은 밑바닥 max는 윗바닥을 가리킴

In [60]:
for r in result :
  print(r)

[{'score': 0.9988903403282166, 'label': 'dog', 'box': {'xmin': 430, 'ymin': 423, 'xmax': 533, 'ymax': 597}}, {'score': 0.9998466968536377, 'label': 'person', 'box': {'xmin': 531, 'ymin': 158, 'xmax': 673, 'ymax': 581}}]
[{'score': 0.9981694221496582, 'label': 'cat', 'box': {'xmin': 541, 'ymin': 122, 'xmax': 719, 'ymax': 535}}, {'score': 0.9980829954147339, 'label': 'cat', 'box': {'xmin': 198, 'ymin': 48, 'xmax': 373, 'ymax': 459}}, {'score': 0.9971736669540405, 'label': 'cat', 'box': {'xmin': 0, 'ymin': 89, 'xmax': 255, 'ymax': 535}}, {'score': 0.8180078864097595, 'label': 'bench', 'box': {'xmin': 218, 'ymin': 355, 'xmax': 718, 'ymax': 535}}, {'score': 0.9972655773162842, 'label': 'cat', 'box': {'xmin': 366, 'ymin': 60, 'xmax': 580, 'ymax': 479}}]
[{'score': 0.9956639409065247, 'label': 'cell phone', 'box': {'xmin': 96, 'ymin': 165, 'xmax': 136, 'ymax': 236}}, {'score': 0.9919517040252686, 'label': 'tv', 'box': {'xmin': 147, 'ymin': 28, 'xmax': 429, 'ymax': 240}}, {'score': 0.997536420